In [ ]:
"""
energy_forecasting.py

Multi-Output Probabilistic LSTM for Energy Grid Intelligence
and Carbon Accounting.

Architecture:

    Historical Grid Data
            |
            v
    Feature Projection
            |
            v
    Shared LSTM Encoder
            |
            v
    Horizon Embeddings
            |
            v
    LSTM Decoder
            |
       +----+----+
       |         |
       v         v
    Demand    Generation
      Head       Head
       |         |
       +----+----+
            |
            v
      Reconciliation
            |
            v
       Carbon Engine
            |
       +----+----+
       |         |
       v         v
   Emissions  Intensity


Input:
    [batch, history_length, input_features]

Outputs:

    demand:
        [batch, horizon, 3]

    generation:
        [batch, horizon, sources, 3]

Quantiles:

    0 -> P10
    1 -> P50
    2 -> P90

This implementation is intended as a production-oriented V1
forecasting backbone. Final probabilistic calibration of derived
carbon metrics should be performed downstream using predictive
sampling and/or conformal calibration.
"""

from __future__ import annotations

from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Constants
# ============================================================

QUANTILES = (0.10, 0.50, 0.90)

P10 = 0
P50 = 1
P90 = 2


# ============================================================
# Monotonic Quantile Head
# ============================================================


class MonotonicQuantileHead(nn.Module):
    """
    Produces ordered non-negative P10/P50/P90 predictions.

    Output:
        [..., 3]

    Guarantees:

        0 <= P10 <= P50 <= P90

    This avoids quantile crossing during inference.
    """

    def __init__(self, hidden_size: int):
        super().__init__()

        self.base = nn.Linear(hidden_size, 1)

        self.delta_p50 = nn.Linear(hidden_size, 1)

        self.delta_p90 = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # ----------------------------------------------------
        # P10
        # ----------------------------------------------------

        p10 = F.softplus(self.base(x))

        # ----------------------------------------------------
        # P10 -> P50
        # Positive distance guarantees P50 >= P10
        # ----------------------------------------------------

        delta_p50 = F.softplus(self.delta_p50(x))

        p50 = p10 + delta_p50

        # ----------------------------------------------------
        # P50 -> P90
        # Positive distance guarantees P90 >= P50
        # ----------------------------------------------------

        delta_p90 = F.softplus(self.delta_p90(x))

        p90 = p50 + delta_p90

        return torch.cat([p10, p50, p90], dim=-1)


# ============================================================
# Generation Quantile Head
# ============================================================


class GenerationQuantileHead(nn.Module):
    """
    Predicts non-negative P10/P50/P90 generation forecasts
    for every generation source.

    Output:

        [batch, horizon, sources, 3]

    Guarantees:

        0 <= P10 <= P50 <= P90
    """

    def __init__(
        self,
        hidden_size: int,
        n_sources: int,
    ):
        super().__init__()

        self.n_sources = n_sources

        self.base = nn.Linear(hidden_size, n_sources)

        self.delta_p50 = nn.Linear(hidden_size, n_sources)

        self.delta_p90 = nn.Linear(hidden_size, n_sources)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # ----------------------------------------------------
        # P10
        # ----------------------------------------------------

        p10 = F.softplus(self.base(x))

        # ----------------------------------------------------
        # P50
        # ----------------------------------------------------

        delta_p50 = F.softplus(self.delta_p50(x))

        p50 = p10 + delta_p50

        # ----------------------------------------------------
        # P90
        # ----------------------------------------------------

        delta_p90 = F.softplus(self.delta_p90(x))

        p90 = p50 + delta_p90

        # [B, H, S] -> [B, H, S, 3]

        return torch.stack([p10, p50, p90], dim=-1)


# ============================================================
# Main Energy Forecasting Model
# ============================================================


class EnergyForecastLSTM(nn.Module):
    """
    Multi-Output Multi-Horizon Probabilistic LSTM.

    Forecasts:

        1. Total electricity demand
        2. Generation mix by source

    Quantiles:

        P10
        P50
        P90

    --------------------------------------------------------
    Input
    --------------------------------------------------------

    x:
        [batch, history_length, input_features]

    Example:

        [32, 336, 40]

    where:

        32  = batch size
        336 = 7 days of 30-minute observations
        40  = number of input features

    --------------------------------------------------------
    Output
    --------------------------------------------------------

    demand:
        [batch, horizon, 3]

    generation:
        [batch, horizon, sources, 3]

    Example:

        demand:
            [32, 48, 3]

        generation:
            [32, 48, 4, 3]

    --------------------------------------------------------
    Notes
    --------------------------------------------------------

    This is a direct multi-horizon forecasting architecture.

    The decoder does not recursively consume previous predictions.
    Instead, each future step is generated from the historical
    context combined with a learned horizon embedding.
    """

    def __init__(
        self,
        input_features: int,
        hidden_size: int = 128,
        num_layers: int = 2,
        horizon: int = 48,
        dropout: float = 0.2,
        generation_sources: int = 4,
    ):
        super().__init__()

        self.input_features = input_features
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.horizon = horizon
        self.dropout_rate = dropout
        self.generation_sources = generation_sources

        # ----------------------------------------------------
        # Input Projection
        # ----------------------------------------------------

        self.input_projection = nn.Sequential(
            nn.Linear(input_features, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
        )

        # ----------------------------------------------------
        # Shared Temporal Encoder
        # ----------------------------------------------------

        self.encoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )

        # ----------------------------------------------------
        # Learnable Future Horizon Embedding
        #
        # One embedding for every forecast step.
        #
        # Example for 48-step forecast:
        #
        #   embedding[0]  -> +30 minutes
        #   embedding[1]  -> +60 minutes
        #   ...
        #   embedding[47] -> +24 hours
        # ----------------------------------------------------

        self.horizon_embedding = nn.Parameter(torch.randn(1, horizon, hidden_size))

        # ----------------------------------------------------
        # Future Decoder
        # ----------------------------------------------------

        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Demand Prediction Head
        # ----------------------------------------------------

        self.demand_head = MonotonicQuantileHead(hidden_size=hidden_size)

        # ----------------------------------------------------
        # Generation Prediction Head
        # ----------------------------------------------------

        self.generation_head = GenerationQuantileHead(
            hidden_size=hidden_size,
            n_sources=generation_sources,
        )

    # ========================================================
    # Encoder
    # ========================================================

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """
        Encode historical time-series data.

        Args:
            x:
                [B, T, F]

        Returns:
            context:
                [B, H]
        """

        # [B, T, F]
        x = self.input_projection(x)

        # [B, T, H]
        encoded, _ = self.encoder(x)

        # Final historical representation
        #
        # [B, H]
        context = encoded[:, -1, :]

        return context

    # ========================================================
    # Forward
    # ========================================================

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass.

        Args:
            x:
                [B, history_length, input_features]

        Returns:
            {
                "demand":
                    [B, horizon, 3],

                "generation":
                    [B, horizon, sources, 3]
            }
        """

        batch_size = x.size(0)

        # ----------------------------------------------------
        # Encode history
        # ----------------------------------------------------

        context = self.encode(x)

        # [B, 1, H]
        context = context.unsqueeze(1)

        # ----------------------------------------------------
        # Expand horizon embeddings
        # ----------------------------------------------------

        horizon_embedding = self.horizon_embedding.expand(batch_size, -1, -1)

        # [B, horizon, H]
        #
        # Every future step receives:
        #
        # historical context
        # +
        # future horizon position
        #

        decoder_input = context + horizon_embedding

        # ----------------------------------------------------
        # Decode future
        # ----------------------------------------------------

        decoded, _ = self.decoder(decoder_input)

        decoded = self.dropout(decoded)

        # ----------------------------------------------------
        # Demand
        # ----------------------------------------------------

        demand = self.demand_head(decoded)

        # ----------------------------------------------------
        # Generation Mix
        # ----------------------------------------------------

        generation = self.generation_head(decoded)

        return {
            "demand": demand,
            "generation": generation,
        }


# ============================================================
# Quantile Loss
# ============================================================


class QuantileLoss(nn.Module):
    """
    Pinball loss for P10/P50/P90 forecasting.
    """

    def __init__(self, quantiles: Tuple[float, ...] = QUANTILES):
        super().__init__()

        self.register_buffer("quantiles", torch.tensor(quantiles, dtype=torch.float32))

    def forward(
        self,
        prediction: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:

            prediction:
                [..., 3]

            target:
                [...]

        Returns:

            scalar loss
        """

        # [..., 1]
        target = target.unsqueeze(-1)

        # [..., 3]
        error = target - prediction

        # [3]
        q = self.quantiles

        loss = torch.maximum(q * error, (q - 1.0) * error)

        return loss.mean()


# ============================================================
# Multi-Output Training Loss
# ============================================================


class EnergyForecastLoss(nn.Module):
    """
    Combined loss for:

        demand forecasting
        generation forecasting
    """

    def __init__(
        self,
        demand_weight: float = 1.0,
        generation_weight: float = 1.0,
    ):
        super().__init__()

        self.quantile_loss = QuantileLoss()

        self.demand_weight = demand_weight
        self.generation_weight = generation_weight

    def forward(
        self,
        prediction: Dict[str, torch.Tensor],
        demand_target: torch.Tensor,
        generation_target: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """
        Args:

            prediction["demand"]:
                [B, H, 3]

            demand_target:
                [B, H]

            prediction["generation"]:
                [B, H, S, 3]

            generation_target:
                [B, H, S]

        Returns:

            {
                "total_loss": ...,
                "demand_loss": ...,
                "generation_loss": ...
            }
        """

        # ----------------------------------------------------
        # Demand loss
        # ----------------------------------------------------

        demand_loss = self.quantile_loss(prediction["demand"], demand_target)

        # ----------------------------------------------------
        # Generation loss
        #
        # Flatten source dimension so quantile loss can operate
        # on [B, H, S, Q].
        # ----------------------------------------------------

        generation_loss = self.quantile_loss(
            prediction["generation"], generation_target
        )

        # ----------------------------------------------------
        # Total
        # ----------------------------------------------------

        total_loss = (
            self.demand_weight * demand_loss + self.generation_weight * generation_loss
        )

        return {
            "total_loss": total_loss,
            "demand_loss": demand_loss,
            "generation_loss": generation_loss,
        }


# ============================================================
# Generation / Demand Reconciliation
# ============================================================


class EnergyReconciler:
    """
    Reconciles predicted generation with predicted demand.

    The model independently forecasts demand and generation.

    Because these quantities may not naturally sum to the same
    value, this layer scales the generation mix proportionally
    so that:

        sum(generation) == demand

    This preserves the predicted generation composition while
    enforcing an energy-balance constraint.

    IMPORTANT:

    This is a practical V1 reconciliation strategy.

    For a more advanced system, optimization-based or
    probabilistic reconciliation can replace this layer.
    """

    def __init__(
        self,
        source_names: Tuple[str, ...],
    ):
        self.source_names = source_names

    def reconcile(
        self,
        demand: torch.Tensor,
        generation: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:

            demand:
                [B, H, Q]

            generation:
                [B, H, S, Q]

        Returns:

            reconciled_generation:
                [B, H, S, Q]
        """

        # ----------------------------------------------------
        # Total predicted generation
        # ----------------------------------------------------

        generation_total = generation.sum(dim=2)

        # ----------------------------------------------------
        # Avoid division by zero
        # ----------------------------------------------------

        generation_total = torch.clamp(generation_total, min=1e-6)

        # ----------------------------------------------------
        # Scale generation to match demand
        # ----------------------------------------------------

        scale = demand / generation_total

        # [B, H, Q]
        #
        # ->
        #
        # [B, H, 1, Q]

        scale = scale.unsqueeze(2)

        reconciled_generation = generation * scale

        return reconciled_generation


# ============================================================
# Carbon Engine
# ============================================================


class CarbonEngine:
    """
    Deterministic carbon accounting engine.

    Input:

        generation:
            MW

        demand:
            MW

    Output:

        emissions:
            kgCO2e

        carbon intensity:
            gCO2e / kWh

    Emission factors must be supplied in:

        gCO2e / kWh
    """

    def __init__(
        self,
        emission_factors: Dict[str, float],
        interval_hours: float = 0.5,
    ):
        """
        Args:

            emission_factors:

                {
                    "coal": 850.0,
                    "gas": 400.0,
                    "wind": 10.0,
                    "solar": 40.0
                }

            interval_hours:

                0.5 for 30-minute forecasts
                1.0 for hourly forecasts
                0.25 for 15-minute forecasts
        """

        self.source_names = tuple(emission_factors.keys())

        self.interval_hours = interval_hours

        self.emission_factors = emission_factors

    def calculate(
        self,
        generation_mw: torch.Tensor,
        demand_mw: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """
        Args:

            generation_mw:
                [B, H, S, Q]

            demand_mw:
                [B, H, Q]

        Returns:

            emissions_kg:
                [B, H, Q]

            carbon_intensity:
                [B, H, Q]
        """

        # ----------------------------------------------------
        # Create emission-factor tensor
        # ----------------------------------------------------

        factors = torch.tensor(
            [self.emission_factors[source] for source in self.source_names],
            dtype=generation_mw.dtype,
            device=generation_mw.device,
        )

        # [S]
        #
        # ->
        #
        # [1, 1, S, 1]

        factors = factors.view(1, 1, -1, 1)

        # ----------------------------------------------------
        # Convert generation MW -> kWh
        #
        # MW × hours × 1000 = kWh
        # ----------------------------------------------------

        energy_kwh = generation_mw * self.interval_hours * 1000.0

        # ----------------------------------------------------
        # Emissions
        #
        # kWh × gCO2e/kWh = gCO2e
        # ----------------------------------------------------

        emissions_grams = (energy_kwh * factors).sum(dim=2)

        # ----------------------------------------------------
        # grams -> kilograms
        # ----------------------------------------------------

        emissions_kg = emissions_grams / 1000.0

        # ----------------------------------------------------
        # Demand MW -> kWh
        # ----------------------------------------------------

        demand_kwh = demand_mw * self.interval_hours * 1000.0

        # ----------------------------------------------------
        # Carbon intensity
        #
        # gCO2e / kWh
        # ----------------------------------------------------

        carbon_intensity = emissions_grams / torch.clamp(demand_kwh, min=1.0)

        return {
            "emissions_kg": emissions_kg,
            "carbon_intensity": carbon_intensity,
        }


# ============================================================
# Forecasting Pipeline
# ============================================================


class EnergyForecastPipeline:
    """
    Combines:

        1. Neural forecasting
        2. Generation reconciliation
        3. Carbon calculation

    NOTE:

    The direct P10/P50/P90 outputs of emissions and carbon
    intensity should be treated as quantile-aligned scenarios,
    not mathematically exact transformed quantiles.

    For production uncertainty propagation, use predictive
    sampling and/or conformal calibration downstream.
    """

    def __init__(
        self,
        model: EnergyForecastLSTM,
        reconciler: EnergyReconciler,
        carbon_engine: CarbonEngine,
    ):
        self.model = model
        self.reconciler = reconciler
        self.carbon_engine = carbon_engine

    @torch.no_grad()
    def predict(
        self,
        historical_input: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """
        Run complete forecasting pipeline.

        Returns:

            {
                "demand_mw": ...,
                "generation_mw": ...,
                "emissions_kg": ...,
                "carbon_intensity_gco2e_per_kwh": ...
            }
        """

        self.model.eval()

        # ----------------------------------------------------
        # Neural forecast
        # ----------------------------------------------------

        prediction = self.model(historical_input)

        demand = prediction["demand"]

        generation = prediction["generation"]

        # ----------------------------------------------------
        # Reconcile generation with demand
        # ----------------------------------------------------

        generation = self.reconciler.reconcile(
            demand=demand,
            generation=generation,
        )

        # ----------------------------------------------------
        # Carbon calculations
        # ----------------------------------------------------

        carbon = self.carbon_engine.calculate(
            generation_mw=generation,
            demand_mw=demand,
        )

        return {
            "demand_mw": demand,
            "generation_mw": generation,
            "emissions_kg": carbon["emissions_kg"],
            "carbon_intensity_gco2e_per_kwh": carbon["carbon_intensity"],
        }


# ============================================================
# Utility: Tensor -> Python Quantile Dictionary
# ============================================================


def quantile_dict(
    values: torch.Tensor,
) -> Dict[str, float]:
    """
    Convert a tensor containing [P10, P50, P90]
    into a JSON-friendly dictionary.

    Expected input:

        [3]

    Returns:

        {
            "p10": ...,
            "p50": ...,
            "p90": ...
        }
    """

    values = values.detach().cpu()

    return {
        "p10": float(values[P10]),
        "p50": float(values[P50]),
        "p90": float(values[P90]),
    }


# ============================================================
# Utility: Convert Forecast Tensor to JSON-Compatible Structure
# ============================================================


def build_forecast_json(
    result: Dict[str, torch.Tensor],
    timestamps,
    model_name: str,
    model_version: str,
    region: str,
    source_names: Tuple[str, ...],
    interval_minutes: int = 30,
) -> Dict:

    demand = result["demand_mw"][0]

    generation = result["generation_mw"][0]

    emissions = result["emissions_kg"][0]

    carbon_intensity = result["carbon_intensity_gco2e_per_kwh"][0]

    forecasts = []

    for h, timestamp in enumerate(timestamps):
        # ----------------------------------------------------
        # Demand
        # ----------------------------------------------------

        demand_json = quantile_dict(demand[h])

        # ----------------------------------------------------
        # Generation
        # ----------------------------------------------------

        generation_json = {}

        for source_idx, source in enumerate(source_names):
            generation_json[source] = quantile_dict(generation[h, source_idx])

        # ----------------------------------------------------
        # Emissions
        # ----------------------------------------------------

        emissions_json = quantile_dict(emissions[h])

        # ----------------------------------------------------
        # Carbon Intensity
        # ----------------------------------------------------

        carbon_json = quantile_dict(carbon_intensity[h])

        forecasts.append(
            {
                "timestamp": str(timestamp),
                "demand_mw": demand_json,
                "generation_mw": generation_json,
                "emissions": {
                    "unit": "kgCO2e",
                    **emissions_json,
                },
                "carbon_intensity": {
                    "unit": "gCO2e_per_kWh",
                    **carbon_json,
                },
            }
        )

    return {
        "forecast_horizon_hours": (len(timestamps) * interval_minutes / 60),
        "interval_minutes": interval_minutes,
        "region": region,
        "model": {
            "name": model_name,
            "version": model_version,
            "quantiles": [
                "P10",
                "P50",
                "P90",
            ],
        },
        "forecasts": forecasts,
    }


# ============================================================
# Example
# ============================================================

if __name__ == "__main__":
    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    INPUT_FEATURES = 32

    HISTORY_LENGTH = 336
    # 336 × 30 minutes = 7 days

    HORIZON = 48
    # 48 × 30 minutes = 24 hours

    GENERATION_SOURCES = (
        "coal",
        "gas",
        "wind",
        "solar",
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = EnergyForecastLSTM(
        input_features=INPUT_FEATURES,
        hidden_size=128,
        num_layers=2,
        horizon=HORIZON,
        dropout=0.2,
        generation_sources=len(GENERATION_SOURCES),
    )

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = EnergyForecastLoss(
        demand_weight=1.0,
        generation_weight=1.0,
    )

    # --------------------------------------------------------
    # Reconciliation
    # --------------------------------------------------------

    reconciler = EnergyReconciler(source_names=GENERATION_SOURCES)

    # --------------------------------------------------------
    # Carbon Factors
    #
    # These values are illustrative.
    # Use your validated/versioned emission factors
    # in the real system.
    # --------------------------------------------------------

    carbon_engine = CarbonEngine(
        emission_factors={
            "coal": 850.0,
            "gas": 400.0,
            "wind": 10.0,
            "solar": 40.0,
        },
        interval_hours=0.5,
    )

    # --------------------------------------------------------
    # Pipeline
    # --------------------------------------------------------

    pipeline = EnergyForecastPipeline(
        model=model,
        reconciler=reconciler,
        carbon_engine=carbon_engine,
    )

    # --------------------------------------------------------
    # Dummy input
    #
    # [batch, history_length, features]
    # --------------------------------------------------------

    x = torch.randn(2, HISTORY_LENGTH, INPUT_FEATURES)

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    prediction = model(x)

    print("=" * 60)
    print("MODEL OUTPUT")
    print("=" * 60)

    print("Demand:", prediction["demand"].shape)

    print("Generation:", prediction["generation"].shape)

    # Expected:

    # Demand:
    # torch.Size([2, 48, 3])

    # Generation:
    # torch.Size([2, 48, 4, 3])

    # --------------------------------------------------------
    # Dummy training targets
    #
    # In a real system these come from your curated
    # historical dataset.
    # --------------------------------------------------------

    demand_target = torch.rand(2, HORIZON) * 10000

    generation_target = torch.rand(2, HORIZON, len(GENERATION_SOURCES)) * 3000

    # --------------------------------------------------------
    # Calculate loss
    # --------------------------------------------------------

    losses = criterion(
        prediction=prediction,
        demand_target=demand_target,
        generation_target=generation_target,
    )

    print()
    print("=" * 60)
    print("TRAINING LOSS")
    print("=" * 60)

    print("Total:", losses["total_loss"].item())

    print("Demand:", losses["demand_loss"].item())

    print("Generation:", losses["generation_loss"].item())

    # --------------------------------------------------------
    # Full inference pipeline
    # --------------------------------------------------------

    result = pipeline.predict(x)

    print()
    print("=" * 60)
    print("PIPELINE OUTPUT")
    print("=" * 60)

    print("Demand:", result["demand_mw"].shape)

    print("Generation:", result["generation_mw"].shape)

    print("Emissions:", result["emissions_kg"].shape)

    print("Carbon intensity:", result["carbon_intensity_gco2e_per_kwh"].shape)

    # --------------------------------------------------------
    # Verify generation reconciliation
    # --------------------------------------------------------

    generation_total = result["generation_mw"].sum(dim=2)

    reconciliation_error = (generation_total - result["demand_mw"]).abs().max()

    print()
    print("Maximum reconciliation error:", reconciliation_error.item())

    # --------------------------------------------------------
    # Example timestamps
    # --------------------------------------------------------

    timestamps = [f"forecast_step_{i:02d}" for i in range(HORIZON)]

    # --------------------------------------------------------
    # Build API-style JSON
    #
    # The first batch item is used here as an example.
    # --------------------------------------------------------

    json_output = build_forecast_json(
        result=result,
        timestamps=timestamps,
        model_name="energy-forecast-lstm",
        model_version="v1.0.0",
        region="NEM",
        source_names=GENERATION_SOURCES,
        interval_minutes=30,
    )

    print()
    print("=" * 60)
    print("API OUTPUT EXAMPLE")
    print("=" * 60)

    # Print only the first forecast step
    # to keep the demonstration readable.

    import json

    print(
        json.dumps(
            {
                **json_output,
                "forecasts": json_output["forecasts"][:1],
            },
            indent=2,
        )
    )